In [ ]:
from peft import PeftModel
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the saved model and tokenizer
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Load adapter model
adapter_model_path = "experiments/kaggle_experiments/20250127_112719/"
adapter_model = PeftModel.from_pretrained(model, adapter_model_path)

# Move model to GPU if available
cuda_enabled = torch.cuda.is_available()
device = torch.device("cuda") if cuda_enabled else torch.device("cpu")
adapter_model.to(device)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 896)
        (layers): ModuleList(
          (0-23): 24 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=896, out_features=896, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=896, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=896, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_features

In [6]:
def generate_chat_response(instruction, max_new_tokens=1024, temperature=0.7, top_p=0.9, repetition_penalty=1.2):
    system_prompt = "You are an expert in The Amazing World Of Gumball, creating stories for the show. You are supposed give stories in the given format:Title: <Title>\nHeadline: <Headline>\n[<Scenario>]\n<Character1>:...\n<Character2>:..."
    messages = [ 
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "Tell me a story about The Amazing World Of Gumball:\n"}, 
        {"role": "assistant", "content": "Tell me a story about The Amazing World Of Gumball:\nTitle: The Virus\nHeadline: The Virus Army\n[The episode starts with tons of virus organisms on Gumball's fur, then one of them stands on a hill-like lump of fur]\nVirus: Brothers, we've mutated many times over for this moment, but nowwe are ready. Today, we take this body; tomorrow, the REST OF THE WORLD!\n[The virus army starts cheering and raising their fists]\nVirus: None will be safe from our infection! Our glory will be greater than the pox, the plague, and bird flu combined!\n[One virus grunts as he pounds his chest]\nVirus:[Raises his fist]Follow me to the nostril!\nVirus Army: Hoorah!\n[The army follows the leader as he charges, but they all stop when the ground starts to shake]\nVirus: Huh?\n[Scene changes to the school corridor where Teri is forcing Gumball to wash his hand; Darwin stands nearby, watching them]\nGumball:[Struggling]Get off! Why don't you clean your own hand, you clean freak?!\nTeri: Wash it now, you disgusting biohazard!\nGumball:[To Darwin]Man, for a paper girl she's surprisingly strong.\nDarwin: I think it's more like you're surprisingly weak.\nGumball:[To Teri]I don't want to wash it![Continues to resist Teri's efforts]\nTeri:Whatis your problem?!\nDarwin:[Sighs]He got a high five from Penny three months ago and refused to wash it since.\nTeri:[Shocked]Don't tell me you haven't showered in ninety days.\nGumball: Of course I've showered, I'm not an art student. I just wrap my hand in swimming trunks to keep it dry.\nDarwin: Ask him how often he washes those.\nGumball: You don't wash swimming trunks, they clean themselves when you swim."}, 
        {"role": "user", "content": instruction}, 
    ]
    # messages = [{"role": "system", "content": "You are a 'The Amazing World Of Gumball' episode generating AI assistant"}, {"role": "user", "content": instruction}]
    input_text=tokenizer.apply_chat_template(messages, tokenize=False)
    inputs = tokenizer.encode_plus(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1536,
    ).to(device)
    
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    
    outputs = adapter_model.generate(
        input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        do_sample=True
    )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [7]:
def generate_response(instruction, max_new_tokens=1024, temperature=0.7, top_p=0.9, repetition_penalty=1.2):
    input_ids = tokenizer.encode(instruction, return_tensors="pt").to(device)
    
    outputs = adapter_model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,  # Adjust temperature
        top_p=top_p,  # Use top-p sampling
        repetition_penalty=repetition_penalty,  # Apply repetition penalty
        do_sample=True
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

In [16]:
from IPython.display import clear_output

base_instruction = "Tell me a story about The Amazing World of Gumball with keywords: gumball, darwin, going to school, school, car ride\nTitle: The School\n"
response = generate_response(base_instruction, max_new_tokens=1024, temperature=0.1)
for i in range(1, 4):
    clear_output(wait=True)
    print(response)
    response = generate_response(response, max_new_tokens=1024, temperature=0.1)

Tell me a story about The Amazing World of Gumball with keywords: gumball, darwin, going to school, school, car ride
Title: The School
Headline:
Gumball and Darwin go to school.
[The scene opens in the living room.]
Darwin: [Sighs] I'm so tired! It's been like this for two weeks now!
Gumball: That sounds awful. What are you doing?
Darwin: I just want some sleep.
[Gumball sighs again as he gets out of bed. He walks over to his desk and picks up an old book from under it. He reads through it while Darwin sits on the couch next to him. They both look at each other before they continue reading. ]
Darwin: So what do we have today? We're learning about...
Gumball: Oh no! You know how much time we've had already spent here without us having any homework or assignments? How many times did that happen last year when we were all younger kids? And then there was the whole "I don't care" thing where we'd sit around talking about nothing but our problems until someone finally said something product